In [2]:
import duckdb

con = duckdb.connect("../data/airhealth_usa.duckdb", read_only=True)
con.execute("SELECT * FROM mart_state_daily_aqi LIMIT 5").df()


,year,month,state_code,state_name,county_code,county_name,avg_aqi,max_aqi,min_aqi,num_days_reporting
0,2025,4,04,Arizona,012,La Paz,53.9,74,25,30
1,2021,9,05,Arkansas,113,Polk,47.2,87,25,30
2,2025,9,06,California,007,Butte,33.0,48,18,5
3,2022,9,13,Georgia,059,Clarke,42.7,62,24,30
4,2023,5,18,Indiana,163,Vanderburgh,57.6,101,36,31


In [3]:
con.execute("SELECT * FROM mart_county_monthly_aqi LIMIT 5").df()


,year,month,state_code,state_name,county_code,county_name,avg_aqi,max_aqi,min_aqi,num_days_reporting
0,2023,9,09,Connecticut,007,Middlesex,38.3,87,24,30
1,2024,11,12,Florida,103,Pinellas,46.6,65,36,30
2,2022,6,13,Georgia,153,Houston,50.1,67,32,30
3,2021,3,17,Illinois,089,Kane,40.2,65,23,31
4,2022,3,21,Kentucky,139,Livingston,39.7,64,24,31


In [4]:
df = con.execute("SELECT * FROM mart_state_daily_aqi").df()
df.describe()

,date_day,avg_aqi,max_aqi,num_counties_reporting
count,91750,91750.00000,91750.000000,91750.000000
mean,2023-05-11 12:34:02.524250,41.87736,64.537929,16.551793
min,2021-01-01 00:00:00,0.00000,0.000000,1.000000
25%,2022-03-07 00:00:00,32.80000,47.000000,9.000000
50%,2023-05-11 00:00:00,40.30000,57.000000,14.000000
75%,2024-07-14 00:00:00,48.70000,71.000000,23.000000
max,2025-11-13 00:00:00,273.10000,8368.000000,53.000000
std,NaN,14.36330,58.103417,11.134557


In [5]:
df.sort_values("max_aqi", ascending=False).head(10)

,date_day,state_code,state_name,avg_aqi,max_aqi,num_counties_reporting,worst_county,defining_pollutant,overall_category
56286,2022-06-12,06,California,197.6,8368,52,Mono,Particulate Matter 10,Hazardous
55437,2022-05-06,06,California,195.2,7835,52,Mono,Particulate Matter 10,Hazardous
10144,2022-04-11,06,California,159.0,3404,52,Mono,Particulate Matter 10,Hazardous
47713,2021-04-13,06,California,111.1,2971,53,Mono,Particulate Matter 10,Hazardous
11650,2022-06-17,06,California,99.7,2889,52,Mono,Particulate Matter 10,Hazardous
74037,2022-11-01,06,California,108.3,2447,51,Mono,Particulate Matter 10,Hazardous
64503,2025-03-18,48,Texas,104.4,2122,42,El Paso,Particulate Matter 10,Hazardous
1603,2022-05-08,06,California,101.9,2050,52,Mono,Particulate Matter 10,Hazardous
88786,2023-05-08,06,California,76.3,1829,52,Mono,Particulate Matter 10,Hazardous
9737,2022-05-28,06,California,76.7,1683,52,Riverside,Particulate Matter 10,Hazardous


In [6]:
df[df["worst_county"] == "Mono"]["date_day"].dt.month.value_counts().sort_index()

date_day
2     3
3     2
4     7
5     6
6     6
9     1
10    2
11    4
12    1
Name: count, dtype: int64

In [7]:
pollutant_df = con.execute(
    """
    SELECT
        p.pollutant_code,
        dd.month,
        AVG(f.aqi) AS avg_aqi
    FROM fact_air_quality_measurement f
    JOIN dim_pollutant p ON f.pollutant_key = p.pollutant_key
    JOIN dim_date dd ON f.date_key = dd.date_key
    WHERE p.pollutant_code IN ('PM2.5', 'O3')
    GROUP BY p.pollutant_code, dd.month
    ORDER BY p.pollutant_code, dd.month
    """
).df()

pollutant_df.pivot(index="month", columns="pollutant_code", values="avg_aqi")

pollutant_code,O3,PM2.5
month,,
1,33.251932,42.833094
2,37.220489,42.332203
3,41.227976,41.677886
4,45.236952,40.464653
5,46.379014,40.637586
6,51.167291,46.905807
7,48.928421,49.803817
8,45.507795,50.004724
9,42.621717,44.913130
